In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q timm>=0.9.0 albumentations>=1.3.0 rich

In [ ]:
import json
import re

print("Remapping cv_splits/cv_splits_3fold.json for Kaggle...")

with open('cv_splits/cv_splits_3fold.json', 'r') as f:
    data = json.load(f)

# Prefixes to strip (case-insensitive)
LOCAL_BASE_REGEX  = re.compile(r"c:/open source/leukiemea/c-nmc_dataset/", re.IGNORECASE)
PKG_FOLDER_REGEX  = re.compile(r"PKG - C-NMC 2019/", re.IGNORECASE)
KAGGLE_PREFIX     = "/kaggle/input/c-nmc-2019-dataset/"

def remap_path(p):
    p_f = p.replace('\\', '/')          # normalise slashes
    res = LOCAL_BASE_REGEX.sub('', p_f) # strip Windows base
    res = PKG_FOLDER_REGEX.sub('', res) # strip intermediate folder
    res = res.lstrip('/')               # remove any stray leading slash
    return KAGGLE_PREFIX + res

for fold_key in data['folds']:
    data['folds'][fold_key]['train_images'] = [
        [remap_path(pair[0]), pair[1]] for pair in data['folds'][fold_key]['train_images']
    ]
    data['folds'][fold_key]['val_images'] = [
        [remap_path(pair[0]), pair[1]] for pair in data['folds'][fold_key]['val_images']
    ]

OUT_JSON = '/kaggle/working/cv_splits_kaggle.json'
with open(OUT_JSON, 'w') as f:
    json.dump(data, f, indent=4)

print(f'Saved to {OUT_JSON}')
print('\nFirst 5 remapped paths (Fold 1 Train):')
for pair in data['folds']['fold_1']['train_images'][:5]:
    print(pair[0])


In [ ]:
import os
import json

print("Spot checking 5 paths from the remapped JSON to ensure they exist on disk...")
with open('/kaggle/working/cv_splits_kaggle.json', 'r') as f:
    check_data = json.load(f)

sample_paths = [pair[0] for pair in check_data['folds']['fold_1']['train_images'][:5]]
all_passed = True

for p in sample_paths:
    exists = os.path.exists(p)
    status = "PASS" if exists else "FAIL"
    print(f"[{status}] {p}")
    if not exists:
        all_passed = False

if not all_passed:
    raise FileNotFoundError("One or more sample paths do not exist. Please check the dataset mount path in Kaggle (/kaggle/input/c-nmc-2019-dataset/) and ensure it matches the script.")
else:
    print("All sample paths verified successfully!")

In [ ]:
!python training_scripts/train_base.py \
  --model effb4 \
  --fold 1 \
  --run_name effb4_cnmc_kaggle \
  --splits_json /kaggle/working/cv_splits_kaggle.json \
  --output_root /kaggle/working/outputs \
  --res 320 \
  --batch_size 48 \
  --num_workers 2 \
  --epochs 150 \
  --patience 25 \
  --lr_backbone 1e-5 \
  --lr_head 1e-4 \
  --no_live

In [ ]:
!python training_scripts/train_base.py \
  --model effb4 \
  --fold 2 \
  --run_name effb4_cnmc_kaggle \
  --splits_json /kaggle/working/cv_splits_kaggle.json \
  --output_root /kaggle/working/outputs \
  --res 320 \
  --batch_size 48 \
  --num_workers 2 \
  --epochs 150 \
  --patience 25 \
  --lr_backbone 1e-5 \
  --lr_head 1e-4 \
  --no_live

In [ ]:
!python training_scripts/train_base.py \
  --model effb4 \
  --fold 3 \
  --run_name effb4_cnmc_kaggle \
  --splits_json /kaggle/working/cv_splits_kaggle.json \
  --output_root /kaggle/working/outputs \
  --res 320 \
  --batch_size 48 \
  --num_workers 2 \
  --epochs 150 \
  --patience 25 \
  --lr_backbone 1e-5 \
  --lr_head 1e-4 \
  --no_live

In [ ]:
import glob
import os

print("Looking for saved best checkpoints...")
ckpts = glob.glob("/kaggle/working/outputs/effb4_cnmc_kaggle/**/*_best.pth", recursive=True)

if not ckpts:
    print("WARNING: No checkpoints found!")
else:
    for c in ckpts:
        size_mb = os.path.getsize(c) / (1024 * 1024)
        print(f"Found: {c} - {size_mb:.2f} MB")

In [ ]:
import shutil
import glob
import os

print("Copying checkpoints to root /kaggle/working/ for persistence...")
ckpts = glob.glob("/kaggle/working/outputs/effb4_cnmc_kaggle/**/*_best.pth", recursive=True)
for c in ckpts:
    dest = os.path.join("/kaggle/working/", os.path.basename(c))
    shutil.copy(c, dest)
    print(f"Saved to persistent output: {os.path.basename(dest)}")